[![在 Colab 開啟](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/roberthsu2003/machine_learning/blob/main/%E6%A9%9F%E7%8E%87%E7%B5%B1%E8%A8%88/02_PMF%E6%A6%82%E7%8E%87%E8%B3%AA%E9%87%8F%E5%88%86%E4%BD%88.ipynb) · [回章節首頁](./README.md)

## 這個觀念要幫我們回答什麼？

**核心問題：骰子出現 4 點的機率，和模型說「90% 是貓」是同一種機率嗎？**

- **用在流程哪裡**：分類問題的輸出解讀（單次預測的分佈長什麼樣）。
- **讀完要能做到**：能讀懂離散結果的機率，區分一次二元、一次多類、多次計數三種情境。
- **閱讀方式**：每個程式格前先讀「為什麼做／怎麼看」，再執行；用實際輸出回答「先想一想」。

觀念連結：[機器學習的類型](../名詞解釋/01_機器學習的類型/README.md)、[數據結構](../名詞解釋/02_數據結構/README.md)

# 02｜PMF：每根柱子就是一個機率

**一、隨機變數：把隨機結果寫成數字**

隨機變數是「結果 → 數字」的函數，例如擲骰子 $X \in \{1,..,6\}$、單張圖片是否為貓 $Y \in \{0,1\}$。分兩大家族：

- **離散**：可以一一列出（骰子 6 面、貓／狗／兔 3 類）。機率用 **PMF** 描述。
- **連續**：區間內無限多值（身高 170.001…公分）。機率用 **PDF＋面積** 描述（見 04）。

**二、PMF：柱高就是機率**

機率質量函數 $p(k) = P(X = k)$ 有兩條鐵律：$p(k) \ge 0$，且**全部加起來等於 1**（$\sum_k p(k) = 1$）。公平骰子每根柱 $1/6$，六根合計為 1——圖上的**面積不負責定義機率**，和連續的 PDF 正好相反。

**三、一次預測的三種長相（本篇三格圖的對照）**

- **伯努利 Bernoulli(p)**：一次二元結果，$P(1)=p$、$P(0)=1-p$。例：單張圖片「貓 90%／非貓 10%」。
- **類別分布 Categorical**：一次多類結果，各類互斥且合計為 1。例：「貓 70%／狗 20%／兔 10%」。只有兩類時它退化成伯努利；三類以上是多項分布做一次試驗的特例。
- **二項 Binomial(n,p)**：**重複 n 次**二元試驗後「成功幾次」（見 03）。它算的是計數 $k=0..n$，不是單張圖片的多類機率——這是學生最常混淆的一條線。

**四、理論 vs 抽樣**：PMF 是理論承諾（骰子每面 1/6）；實際擲 30 次不會剛好每面 5 次，樣本多通常更穩定，但**一次大樣本也不保證每根柱都更貼理論**。90% 是模型對「這一張」的估計，不是保證；要當可靠機率用，需在獨立資料上做校準檢查。多標籤任務可同時屬多類，不能套用合計為 1 的規則。

本篇全部使用教學合成數字與已知理論。

## 準備共用繪圖設定

**為什麼做**：統一繁體中文字型與配色，讓讀圖重點清楚。

**怎麼看**：這格只載入工具，沒有訓練模型；會列出實際字型。需要套件時由老師協助準備 NumPy、pandas、SciPy、Matplotlib、Seaborn。

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from IPython.display import display

# 設定中文字型支援
# --- 設定 Matplotlib 中文字型支援（Mac / Windows / Colab 通用）---
import glob
import os
try:
    import mplfonts
except ImportError:  # Colab 環境若缺套件則自動安裝（本機已在 requirements.txt）
    %pip install -q mplfonts
    import mplfonts
from matplotlib import font_manager
from mplfonts import use_font

for f in glob.glob(os.path.join(os.path.dirname(mplfonts.__file__), "fonts", "*.[ot]tf")):
    font_manager.fontManager.addfont(f)   # 直接註冊 mplfonts 內附的字型，不改任何檔案
use_font("Noto Sans CJK SC")              # 套用思源黑體，負號也會一起處理

sns.set_theme(style='whitegrid', rc={
    'axes.unicode_minus': False, 'figure.facecolor': '#F5F7F2',
    'axes.facecolor': '#FFFFFF', 'axes.spines.top': False,
    'axes.spines.right': False, 'grid.alpha': 0.22,
    'axes.titlesize': 14, 'axes.labelsize': 11, 'figure.dpi': 110})
GREEN, ORANGE, PURPLE = '#2F6657', '#C17B45', '#786497'
rng = np.random.default_rng(42)
print('中文字型：Noto Sans CJK SC（mplfonts 通用寫法）')

## 01｜從骰子讀懂 PMF

**為什麼做**：離散結果可以逐一列出，每個可能值對應一個機率。

**怎麼看**：柱高是 P(X=k)，六根柱的機率相加為 1；柱子的畫面面積不負責定義機率。

In [ ]:
faces = np.arange(1, 7)
pmf = np.full(6, 1 / 6)
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(faces, pmf, color=GREEN, width=0.65)
for x, prob in zip(faces, pmf):
    ax.text(x, prob + 0.01, f'{prob:.1%}', ha='center')
ax.set(xticks=faces, ylim=(0, 0.23), xlabel='骰子點數', ylabel='機率',
       title='公平骰子：六種互斥結果，機率合計為 1')
plt.show()
print('擲出至少 4 點的機率：', pmf[faces >= 4].sum())
assert np.isclose(pmf.sum(), 1)

## 02｜單次分類與多次計數分開看

**為什麼做**：伯努利描述一次二元結果；類別分布描述一次多類別結果，這裡各類別互斥且涵蓋全部結果。

**怎麼看**：兩個面板的機率都相加為 1。二項分布是在重複獨立二元試驗後計算成功次數，不是單張圖片的多類別機率。

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
examples = [(['非貓', '貓'], [0.1, 0.9], '伯努利：一次二元分類'),
            (['貓', '狗', '兔'], [0.7, 0.2, 0.1], '類別分布：一次多類別分類')]
for ax, (labels, probs, title) in zip(axes, examples):
    ax.bar(labels, probs, color=[GREEN, ORANGE, PURPLE][:len(probs)])
    for x, prob in enumerate(probs):
        ax.text(x, prob + 0.03, f'{prob:.0%}', ha='center')
    ax.set(title=title, ylabel='模型估計機率', ylim=(0, 1.05))
    assert np.isclose(sum(probs), 1)
plt.show()

## 03｜理論機率與有限次抽樣比例

**為什麼做**：模擬擲骰子，看隨機樣本為何不會每次都完美平均。

**怎麼看**：固定種子方便比較。樣本變多通常能讓比例更穩定，但一次較大的樣本也不保證每根柱都更接近理論。

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=True)
for ax, size in zip(axes, [30, 3000]):
    rolls = np.random.default_rng(42).integers(1, 7, size=size)
    observed = np.bincount(rolls, minlength=7)[1:] / size
    ax.bar(faces, observed, color=GREEN, label='觀察比例')
    ax.axhline(1 / 6, color=ORANGE, linestyle='--', label='理論機率')
    ax.set(title=f'抽樣 {size:,} 次', xticks=faces, xlabel='骰子點數', ylabel='比例')
    ax.legend()
plt.show()

## 接回機器學習

模型寫出 90% 是它對本張圖片的估計，不是保證；要把分數解釋成可靠機率，需要在獨立資料上檢查校準。多標籤任務可同時屬於多類，不能套用本例「全部機率加總為 1」的規則。

## 換你解讀

1. PMF 的柱高和 PDF 的曲線高度有什麼不同？
2. 30 次骰子不是每面 5 次，能直接判定骰子不公平嗎？
3. 單張圖片的三個類別，與十次成功計數有什麼不同？

### 請 AI 幫你做一個小實驗

請 AI 將一個骰面機率提高、其他五面均分剩餘機率。先確認總和為 1，再比較理論 PMF 與抽樣比例。

先說出預期，再看實際圖形，最後寫下「證據 → 解釋 → 不能下的結論」。

[回章節首頁](./README.md)